In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [3]:
# Folder containing .db files
INPUT_FOLDER = Path(r"C:\Users\alrazz\Documents\Anonating files")

# Output file
OUTPUT_FILE = "ID_SP_records.csv"


In [4]:
# ---------------------------------------------------------
# PROCESS DATABASE FILES
# ---------------------------------------------------------

results = []

# Find all .db files in the folder
db_files = list(INPUT_FOLDER.glob("*.db"))

print(f"Found {len(db_files)} .db files.")

for db_file in db_files:
    print(f"Processing: {db_file.name}")

    try:
        # Open database in read-only mode
        connection = sqlite3.connect(f"file:{db_file}?mode=ro", uri=True)

        # Read only the columns we need
        query = """
            SELECT
                id,
                "Turku_NLP",
                "Turku_NLP_sub"
            FROM texts
        """

        df = pd.read_sql_query(query, connection)

        connection.close()

        # -------------------------------------------------
        # EXPLODE Turku_NLP
        # -------------------------------------------------

        # Convert NULLs to empty strings
        df["Turku_NLP"] = df["Turku_NLP"].fillna("")

        # Split on semicolon
        df["Turku_NLP_exploded"] = df["Turku_NLP"].str.split(";")

        # One row per label
        df = df.explode("Turku_NLP_exploded")

        # Remove whitespace
        df["Turku_NLP_exploded"] = (
            df["Turku_NLP_exploded"]
            .str.strip()
            .str.upper()
        )

        # -------------------------------------------------
        # KEEP ONLY ID AND SP
        # -------------------------------------------------

        df = df[df["Turku_NLP_exploded"].isin(["ID", "SP"])].copy()

        # Add filename
        df["filename"] = db_file.name

        # Rename exploded column
        df = df.rename(columns={
            "Turku_NLP_exploded": "matched_label"
        })

        # Keep the columns in a convenient order
        df = df[
            [
                "filename",
                "id",
                "matched_label",
                "Turku_NLP",
                "Turku_NLP_sub"
            ]
        ]

        # Add to overall results
        results.append(df)

    except Exception as e:
        print(f"  ERROR processing {db_file.name}: {e}")

Found 73 .db files.
Processing: ed.db
Processing: ed_2.db
Processing: ed_3.db
Processing: en.db
Processing: en_10.db
Processing: en_11.db
Processing: en_2.db
Processing: en_3.db
Processing: en_4.db
Processing: en_5.db
Processing: en_6.db
Processing: en_7.db
Processing: en_8.db
Processing: en_9.db
Processing: fi.db
Processing: HI_2_clean_dedup.db
Processing: HI_clean_dedup.db
Processing: HI_ID_LY_SP_clean2_dedup.db
Processing: ID_clean_dedup.db
Processing: it.db
Processing: it_2.db
Processing: it_3.db
Processing: lt.db
Processing: lt_2.db
Processing: lt_3.db
Processing: lt_4.db
Processing: LY_clean_dedup.db
Processing: MT.db
Processing: nb.db
Processing: nb_2.db
Processing: nb_3.db
Processing: nb_4.db
Processing: nb_5.db
Processing: nb_6.db
Processing: nb_7.db
Processing: New_HI_ID_LY_OP_SP_clean_clean.db
Processing: ob.db
Processing: ob_2.db
Processing: ob_3.db
Processing: ob_4.db
Processing: oi.db
Processing: oi_2.db
Processing: oi_3.db
Processing: oo.db
Processing: oo_2.db
Processing

In [5]:
# ---------------------------------------------------------
# COMBINE EVERYTHING
# ---------------------------------------------------------

if results:
    final_df = pd.concat(results, ignore_index=True)

    # Save as CSV
    final_df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("Done!")
    print(f"Found {len(final_df)} matching records.")
    print(f"Output saved to:")
    print(OUTPUT_FILE)

else:
    print()
    print("No records containing ID or SP were found.")


Done!
Found 784 matching records.
Output saved to:
ID_SP_records.csv
